**Q1: Build Personalized Knowledge Base**

In [15]:
import pandas as pd

# 4 fixed FAQ entries
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi",
        "category": "billing"
    }
]

# Last two digits of roll number = 8 and 6
# Digit 8: 8 % 3 = 2 -> general
# Digit 6: 6 % 3 = 0 -> billing

custom_entries = [
    {
        "question": "how can i contact customer support",
        "answer": "You can contact customer support through phone or email.",
        "keywords": "support help contact",
        "category": "general"
    },
    {
        "question": "can i get a receipt for my payment",
        "answer": "Yes, a payment receipt will be provided after successful payment.",
        "keywords": "receipt payment bill",
        "category": "billing"
    }
]

# Combine fixed and custom entries
all_entries = fixed_entries + custom_entries

# Create DataFrame
df = pd.DataFrame(all_entries)

# Print final 6-row DataFrame
print(df)

                             question  \
0              what is the annual fee   
1               how to reset password   
2         what are your working hours   
3               how can i pay the fee   
4  how can i contact customer support   
5  can i get a receipt for my payment   

                                              answer              keywords  \
0                          The annual fee is Rs 500.       fee cost charge   
1                   Go to Settings > Reset Password.  password reset login   
2                          We are open 9 AM to 5 PM.     hours timing open   
3         You can pay via UPI, card, or net banking.       pay payment upi   
4  You can contact customer support through phone...  support help contact   
5  Yes, a payment receipt will be provided after ...  receipt payment bill   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  general  
5  billing  


**Q2: Generate and Score a Hypothesis**

In [16]:
# Function to generate and score hypotheses
def score_hypothesis(query, df):

    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():

        # Combine question and keywords
        text = row["question"].lower() + " " + row["keywords"].lower()
        text_words = set(text.split())

        # Find matching words
        matched_words = query_words.intersection(text_words)

        # Calculate confidence
        if len(query_words) > 0:
            confidence = len(matched_words) / len(query_words)
        else:
            confidence = 0

        # Add only matching entries
        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": round(confidence, 2)
            })

    # Rank by confidence
    results.sort(key=lambda x: x["confidence"], reverse=True)

    return pd.DataFrame(results)

# Test the function
query = "how can i pay the fee"
result_df = score_hypothesis(query, df)
print(result_df)

                             question  \
0               how can i pay the fee   
1  how can i contact customer support   
2              what is the annual fee   
3  can i get a receipt for my payment   
4               how to reset password   

                                              answer category  confidence  
0         You can pay via UPI, card, or net banking.  billing        1.00  
1  You can contact customer support through phone...  general        0.50  
2                          The annual fee is Rs 500.  billing        0.33  
3  Yes, a payment receipt will be provided after ...  billing        0.33  
4                   Go to Settings > Reset Password.  account        0.17  


**Q3: Find Questions by Category**

In [17]:
def same_category(category_name, df):

    # Filter rows having the given category
    result = df[df["category"] == category_name]

    # Return only the questions
    return result["question"].tolist()

# Call the function using the category
# of one personalized entry from Q1
category_name = "general"

# Get questions belonging to the category
questions = same_category(category_name, df)

# Print the result
print("Questions in", category_name, "category:")

for question in questions:
    print(question)

Questions in general category:
what are your working hours
how can i contact customer support


**Q4: Add a New Keyword and Save Updated DataFrame**

In [18]:
# Select one entry from the knowledge base
entry_index = 0

# Ask the user to enter a new keyword
new_keyword = input("Enter a new keyword: ")

# Add the new keyword to the selected entry
df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

# Display the updated entry
print("\nUpdated Entry:")
print(df.loc[entry_index])

# Save the complete updated DataFrame to a CSV file
filename = "1024170386_faq_data.csv"
df.to_csv(filename, index=False)

print("\nUpdated DataFrame saved to:", filename)

Enter a new keyword: yearly

Updated Entry:
question       what is the annual fee
answer      The annual fee is Rs 500.
keywords       fee cost charge yearly
category                      billing
Name: 0, dtype: object

Updated DataFrame saved to: 1024170386_faq_data.csv


**Q5: Count FAQ Entries per Category**

In [19]:
# Group the DataFrame by category
category_count = df.groupby("category").size()

# Print the number of entries in each category
print("Number of FAQ entries per category:")
print(category_count)

Number of FAQ entries per category:
category
account    1
billing    3
general    2
dtype: int64


**Q6: Modify Q2 Scoring Function to Handle Ties**

In [20]:
def score_hypothesis(query, df):

    # Convert query into lowercase words
    query_words = set(query.lower().split())

    # Store matching results
    results = []

    # Check each entry in the DataFrame
    for index, row in df.iterrows():

        # Combine question and keywords
        text = row["question"].lower() + " " + row["keywords"].lower()

        # Convert FAQ text into words
        text_words = set(text.split())

        # Find common words
        matched_words = query_words.intersection(text_words)

        # Calculate confidence score
        if len(query_words) > 0:
            confidence = len(matched_words) / len(query_words)
        else:
            confidence = 0

        # Store matching entries
        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": round(confidence, 2)
            })

    # Sort results from highest to lowest confidence
    results.sort(
        key=lambda x: x["confidence"],
        reverse=True
    )

    # Find the highest confidence score
    if results:
        highest_score = results[0]["confidence"]

        # Get all entries having the highest score
        best_matches = [
            result for result in results
            if result["confidence"] == highest_score
        ]

        # Print all highest-scoring entries
        print("\nBest matching entries:")

        for result in best_matches:
            print(result)

        # Check whether there is a tie
        if len(best_matches) > 1:
            print("\nTie detected! Multiple entries have the same highest score.")

    else:
        print("No matching entries found.")

# ------------------------------------------------
# Demonstration 1: Query that produces a tie
# ------------------------------------------------

print("DEMONSTRATION 1")
query = "fee"
score_hypothesis(query, df)

# ------------------------------------------------
# Demonstration 2: Query that does not produce a tie
# ------------------------------------------------
print("\nDEMONSTRATION 2")
query = "password"
score_hypothesis(query, df)

DEMONSTRATION 1

Best matching entries:
{'question': 'what is the annual fee', 'answer': 'The annual fee is Rs 500.', 'category': 'billing', 'confidence': 1.0}
{'question': 'how can i pay the fee', 'answer': 'You can pay via UPI, card, or net banking.', 'category': 'billing', 'confidence': 1.0}

Tie detected! Multiple entries have the same highest score.

DEMONSTRATION 2

Best matching entries:
{'question': 'how to reset password', 'answer': 'Go to Settings > Reset Password.', 'category': 'account', 'confidence': 1.0}
